In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_selection import mutual_info_classif

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")

DATA_DIR = "/kaggle/input/datasets/dhoogla/cicids2017"

PARQUET_FILES = [
    "Benign-Monday-no-metadata.parquet",
    "Botnet-Friday-no-metadata.parquet",
    "Bruteforce-Tuesday-no-metadata.parquet",
    "DDoS-Friday-no-metadata.parquet",
    "DoS-Wednesday-no-metadata.parquet",
    "Infiltration-Thursday-no-metadata.parquet",
    "Portscan-Friday-no-metadata.parquet",
    "WebAttacks-Thursday-no-metadata.parquet",
]

In [ ]:
def infer_attack_category_from_filename(fname: str) -> str:
    base = os.path.basename(fname).lower()
    if "benign" in base:
        return "Normal"
    if "botnet" in base:
        return "Bot"
    if "bruteforce" in base:
        return "BruteForce"
    if "ddos" in base:
        return "DDoS"
    if "dos" in base:
        return "DoS"
    if "portscan" in base:
        return "PortScan"
    if "webattacks" in base or "webattacks" in base:
        return "WebAttack"
    if "infiltration" in base:
        return "Infiltration"
    return "Unknown"

dfs = []
for fname in PARQUET_FILES:
    fpath = os.path.join(DATA_DIR, fname)
    df_part = pd.read_parquet(fpath)
    df_part["attack_category"] = infer_attack_category_from_filename(fname)
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)
df.head()

In [ ]:
print("Shape (rows, columns):", df.shape)

memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Approx. memory usage: {memory_mb:.2f} MB")

print("\nDataFrame info():")
df.info()

numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols].describe().T.head()

In [ ]:
EDA_SAMPLE_SIZE = 500_000
if len(df) > EDA_SAMPLE_SIZE:
    df_eda = df.sample(EDA_SAMPLE_SIZE, random_state=42)
else:
    df_eda = df.copy()

df_eda.shape

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.countplot(
    data=df,
    x="attack_category",
    order=df["attack_category"].value_counts().index,
    palette="tab10"
)

plt.title("Class distribution (attack categories)")
plt.xlabel("Attack category")
plt.ylabel("Count")
plt.xticks(rotation=30, ha="right")

for p in ax.patches:
    height = p.get_height()
    ax.annotate(
        f"{height:,}",
        (p.get_x() + p.get_width() / 2., height),
        ha="center", va="bottom", fontsize=10, rotation=90
    )

plt.tight_layout()
plt.savefig("class_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

df["attack_category"].value_counts(normalize=True).mul(100).round(2)

In [ ]:
# Use the EDA sample to keep things tractable
numeric_cols = df_eda.select_dtypes(include=[np.number]).columns

corr_pearson = df_eda[numeric_cols].corr(method="pearson")
corr_spearman = df_eda[numeric_cols].corr(method="spearman")

corr_pearson.shape, corr_spearman.shape

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(
    corr_pearson,
    cmap="coolwarm",
    center=0,
    square=False,
    cbar_kws={"shrink": 0.6},
)
plt.title("Pearson correlation heatmap (numeric features)")
plt.tight_layout()
plt.savefig("correlation_heatmap_pearson.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(14, 10))
sns.heatmap(
    corr_spearman,
    cmap="coolwarm",
    center=0,
    square=False,
    cbar_kws={"shrink": 0.6},
)
plt.title("Spearman correlation heatmap (numeric features)")
plt.tight_layout()
plt.savefig("correlation_heatmap_spearman.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder

target_col = "attack_category"
X = df_eda[numeric_cols].fillna(0)
y = df_eda[target_col]

le = LabelEncoder()
y_enc = le.fit_transform(y)

mi_scores = mutual_info_classif(X, y_enc, random_state=42)
mi_series = pd.Series(mi_scores, index=numeric_cols).sort_values(ascending=False)

mi_series.head(20)

In [ ]:
TOP_K_MI = 30

plt.figure(figsize=(10, 8))
sns.barplot(
    x=mi_series.head(TOP_K_MI),
    y=mi_series.head(TOP_K_MI).index,
    palette="viridis"
)
plt.title("Top mutual information features w.r.t. attack category")
plt.xlabel("Mutual information")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("mutual_information_top_features.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(
    corr_pearson,
    cmap="coolwarm",
    center=0,
    square=False,
    cbar_kws={"shrink": 0.6},
)
plt.title("Correlation heatmap (Pearson)")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Replace these with the actual column names in your dataset
packet_len_cols = [c for c in df.columns if "Packet Length" in c or "Pkt Len" in c]
flow_duration_col = [c for c in df.columns if "Flow Duration" in c][0]
bytes_per_sec_col = [c for c in df.columns if "Flow Bytes/s" in c or "Bytes/s" in c][0]
pkts_per_sec_col = [c for c in df.columns if "Flow Packets/s" in c or "Pkts/s" in c][0]

packet_len_feature = packet_len_cols[0] if packet_len_cols else None
packet_len_feature

In [ ]:
packet_len_feature = "Average Packet Size"        # example
flow_duration_col = "Flow Duration"
bytes_per_sec_col = "Flow Bytes/s"
pkts_per_sec_col = "Flow Packets/s"

In [ ]:
def plot_feature_by_attack(feature, log_scale=False, filename=None):
    plt.figure(figsize=(12, 6))
    ax = sns.boxplot(
        data=df_eda,
        x="attack_category",
        y=feature,
        order=df["attack_category"].value_counts().index,
        showfliers=False,
        palette="tab10"
    )
    plt.title(f"{feature} distribution by attack category")
    plt.xlabel("Attack category")
    plt.ylabel(feature)
    plt.xticks(rotation=30, ha="right")
    if log_scale:
        ax.set_yscale("log")
    plt.tight_layout()
    if filename is not None:
        plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()

plot_feature_by_attack(flow_duration_col, log_scale=True,
                       filename="flow_duration_by_attack.png")
plot_feature_by_attack(bytes_per_sec_col, log_scale=True,
                       filename="bytes_per_sec_by_attack.png")
plot_feature_by_attack(pkts_per_sec_col, log_scale=True,
                       filename="pkts_per_sec_by_attack.png")
if packet_len_feature is not None:
    plot_feature_by_attack(packet_len_feature, log_scale=True,
                           filename="packet_length_by_attack.png")

In [ ]:
plt.figure(figsize=(12, 6))
sns.violinplot(
    data=df_eda,
    x="attack_category",
    y=bytes_per_sec_col,
    order=df["attack_category"].value_counts().index,
    scale="width",
    cut=0,
    inner="quartile",
    palette="tab10"
)
plt.yscale("log")
plt.title(f"{bytes_per_sec_col} by attack category")
plt.xlabel("Attack category")
plt.ylabel(bytes_per_sec_col)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("bytes_per_sec_violin_by_attack.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Basic counts per category
counts = df["attack_category"].value_counts().rename("count")

# Per-category basic stats for a few core features
core_feats = [flow_duration_col, bytes_per_sec_col, pkts_per_sec_col]
if packet_len_feature is not None:
    core_feats.append(packet_len_feature)

agg_funcs = ["mean", "std", "median", "min", "max"]

stats_per_cat = (
    df.groupby("attack_category")[core_feats]
      .agg(agg_funcs)
)

# Flatten MultiIndex columns
stats_per_cat.columns = [
    f"{feat}_{stat}"
    for feat, stat in stats_per_cat.columns.to_flat_index()
]

eda_report = pd.concat([counts, stats_per_cat], axis=1).reset_index()
eda_report.rename(columns={"index": "attack_category"}, inplace=True)

eda_report.to_csv("eda_report.csv", index=False)
eda_report.head()